# 04 — Scalable processing
BFT65B · Student ID 11230945

Compare native pandas, Polars and Spark transformations on the verified Silver financial panel. Evaluate a filtered aggregation, adjacent-quarter growth and a state–quarter peer join. Measure actual-scale processing, synthetic stress scaling and Polars thread sensitivity. All outcomes and retrospective audit fields remain outside these experiments.

## 1. Repository and runtime
Use a CPU runtime and Java 17 or later. Install the pinned environment before importing the engines. Spark runs locally, not on a distributed cluster.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, shutil
REPO_URL = "https://github.com/NguyenDuong0508/11230945.git"
REPO = Path("/content/11230945")
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    assert subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=REPO, text=True).strip() == REPO_URL
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO, check=True)
print("Git revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())
java = shutil.which("java")
if java is None:
    raise RuntimeError("Java 17 or later is required")
os.environ["JAVA_HOME"] = str(Path(java).resolve().parents[1])
print(subprocess.run([java,"-version"],check=True,capture_output=True,text=True).stderr)
subprocess.run([sys.executable,"-m","pip","install","-r",str(REPO/"requirements.txt")],check=True)
def command(script,*args):
    result = subprocess.run([sys.executable,str(REPO/script),*map(str,args)],cwd=REPO,text=True,capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()

## 2. Mandatory offline validation
Run the complete repository suite, including actual Parquet and Spark integration. Failures and skipped tests block progression.

In [ ]:
import xml.etree.ElementTree as ET
TESTS_PASSED = False
result_path = Path("/content/part3_tests.xml")
result_path.unlink(missing_ok=True)
result = subprocess.run([sys.executable,"-m","pytest","-q","-ra","--tb=short","--show-capture=no",f"--junitxml={result_path}","tests"],cwd=REPO,env=dict(os.environ,PYTEST_DISABLE_PLUGIN_AUTOLOAD="1",SPARK_LOCAL_IP="127.0.0.1",PYSPARK_PYTHON=sys.executable),capture_output=True,text=True)
print(result.stdout)
if result.stderr: print(result.stderr)
if not result_path.is_file(): raise RuntimeError("Missing pytest report")
cases=list(ET.parse(result_path).getroot().iter("testcase"))
problems=[(case,node) for case in cases for node in list(case) if node.tag in ("failure","error","skipped")]
for case,node in problems:
    print(case.get("classname"),case.get("name"),node.tag,node.get("message"),node.text or "")
if result.returncode or not cases or problems: raise RuntimeError("Offline validation failed; inspect diagnostics")
TESTS_PASSED=True
print(f"Mandatory offline validation: PASS ({len(cases)} tests, zero skips)")

## 3. Verified Silver input and experiment scope
Resolve the active panel, check its source lineage and use only its financial partitions. All scales share the same eight-column projection and quarter-wise Snappy layout. Synthetic replicas have disjoint CERTs and preserve within-bank history.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
WORKSPACE = Path("/content/drive/MyDrive/11230945")
SCRATCH = Path("/content/fdic_processing_scratch")
assert TESTS_PASSED
assert not (WORKSPACE/"pending_source.json").exists()
command("run_part2.py","verify","--workspace",WORKSPACE)
pointer=json.loads((WORKSPACE/"active_panel.json").read_text())
SILVER=WORKSPACE/pointer["path"]
manifest=json.loads((SILVER/"manifest.json").read_text())
assert manifest["state"]=="VERIFIED" and manifest["summary"]["quarters"]==170
print("Silver manifest SHA-256:",pointer["manifest_sha256"])
print("Observed rows:",f"{manifest['summary']['panel_rows']:,}")
print("Planned logical rows:",{f"{s}x":s*manifest['summary']['panel_rows'] for s in (1,5,10)})
print("Local disk free (GiB):",round(shutil.disk_usage('/content').free/2**30,2))
print("Protocol: 24 scenarios; 24 warm-ups; 72 measured trials; 15 untimed plan captures")

## 4. Execute the controlled benchmark
Stage and verify the projected files locally; run isolated scenarios. A warm-up precedes three measured invocations per scenario. Spark is run at actual scale; pandas and Polars are measured at all scales. No failed or mismatched scenario is silently omitted.

In [ ]:
assert TESTS_PASSED
# Stream progress and error diagnostics into the saved notebook output.
cmd = [sys.executable,str(REPO/"run_part3.py"),"run","--workspace",str(WORKSPACE),"--scratch",str(SCRATCH)]
with subprocess.Popen(cmd,cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
    for line in process.stdout:
        print(line,end="",flush=True)
    if process.wait()!=0:
        raise RuntimeError("Benchmark failed; inspect the worker diagnostics above")

## 5. Verify and resolve measured evidence
Recompare all saved trial results, check source lineage and checksums, then read the evidence for presentation. Benchmark replicas are not retained as downstream data.

In [ ]:
command("run_part3.py","verify","--workspace",WORKSPACE)
import hashlib
p=json.loads((WORKSPACE/"active_processing.json").read_text())
EVIDENCE=WORKSPACE/p["path"]
with (EVIDENCE/"manifest.json").open("rb") as f:
    assert hashlib.file_digest(f,"sha256").hexdigest()==p["manifest_sha256"]
evidence=json.loads((EVIDENCE/"manifest.json").read_text())
assert evidence["silver_manifest_sha256"]==pointer["manifest_sha256"]
assert evidence["state"]=="VERIFIED" and evidence["measured_trials"]==72
print(json.dumps({k:v for k,v in evidence.items() if k!="files"},indent=2))
print(json.dumps(json.loads((EVIDENCE/"environment.json").read_text()),indent=2))

## 6. Actual-scale performance and result equivalence
Compare the same logical selected rows and output tables. Timing includes read, computation and materialization; startup, checks and plan capture are excluded. Peak RSS is the sampled whole process tree, not net dataframe allocation.

In [ ]:
import pandas as pd
from IPython.display import display,Markdown,Image,HTML

def show(title,df,formats=None):
    display(Markdown(f"**{title}**"))
    display(df.style.hide(axis="index").format(formats or {},na_rep="—"))
    display(HTML('<div style="height:18px"></div>'))
def fig(title,file):
    display(Markdown(f"**{title}**"))
    display(Image(filename=str(EVIDENCE/file),width=680))
    display(HTML('<div style="height:22px"></div>'))
summary=pd.read_csv(EVIDENCE/"summary.csv")
base=summary[(summary.scale==1)&(summary.threads==2)]
show("Table 1. Actual-scale processing performance",base.drop(columns=["scale","threads"]),{"input_rows":"{:,}","median_seconds":"{:.4f}","iqr_seconds":"{:.4f}","median_peak_rss_mib":"{:.1f}","median_incremental_rss_mib":"{:.1f}"})
wide=base.pivot(index="workload",columns="engine",values="median_seconds").reset_index()
wide["pandas_over_polars"]=wide.pandas/wide.polars
wide["spark_over_polars"]=wide.pyspark/wide.polars
show("Table 2. Actual-scale median latency ratios",wide,{x:"{:.3f}" for x in wide.columns if x!="workload"})
fig("Figure 1. Actual-scale median latency by workload","runtime.png")
print("All saved warm-up and measured results passed row accounting, cross-engine parity and replica-scaling checks.")

## 7. Inspect optimized and physical execution plans
Read the recorded plans. Explain projection, filter, aggregation, ordering and join strategy from actual operators; do not infer a distributed network cost from a local Spark exchange. A lazy plan is not proof that every operator streams.

In [ ]:
for workload in ("quarter_summary","adjacent_growth","peer_join"):
    for engine in ("polars","pyspark"):
        display(Markdown(f"**{engine} — {workload}**"))
        plan=(EVIDENCE/"plans"/f"1x_{engine}_{workload}_2t.txt").read_text()
        print(plan)
        lines=[line.strip() for line in plan.splitlines() if any(k in line.lower() for k in ("project","selection","pushedfilters","readschema","exchange","window","shift(","sort","join","aggregate"))]
        if lines:
            print("\nSelected operator evidence:")
            print("\n".join(lines[:12]))
display(Markdown("""**Interpretation.** Projection limits the physical columns required by each workload; the summary date predicate may be pushed into the Parquet scan. The growth workload requires CERT-specific ordering before shifting, and retains the full available history to avoid boundary artifacts. The peer workload builds a unique state–quarter mean table and joins it to observations, preserving row cardinality. Spark aggregation, sorting and joins can introduce exchanges; inspect the actual adaptive plan and join strategy above. Under local execution, exchanges are local redistribution rather than measured network traffic. Polars' optimized lazy plan does not establish out-of-core execution."""))

## 8. Synthetic stress scaling and thread sensitivity
The larger inputs are replicated bank histories, not new institutions. Compare measured time, logical row throughput and sampled memory. The Amdahl fraction is an apparent diagnostic under ideal assumptions; values outside [0,1] invalidate that interpretation. No Spark cluster was measured.

In [ ]:
stage=json.loads((EVIDENCE/"staging.json").read_text())
show("Table 3. Staged benchmark inputs",pd.DataFrame([{"scale":k,"rows":v["rows"],"filtered_rows":v["filtered_rows"],"size_mib":v["bytes"]/2**20,"files":len(v["files"])} for k,v in stage.items()]),{"rows":"{:,}","filtered_rows":"{:,}","size_mib":"{:.2f}"})
show("Table 4. Single-machine scaling",pd.read_csv(EVIDENCE/"scaling.csv"),{"input_rows":"{:,}","time_ratio":"{:.3f}","rows_per_second":"{:,.0f}"})
show("Table 5. Polars thread-count sensitivity",pd.read_csv(EVIDENCE/"parallelism.csv"),{k:"{:.3f}" for k in ("one_thread_s","two_threads_s","speedup","efficiency","apparent_serial_fraction")})
fig("Figure 2. Runtime under synthetic scale-up","scaling.png")
fig("Figure 3. Sampled peak process-tree memory under scale-up","memory.png")
show("Table 6. Measured stress-test runtime and memory",summary[(summary.scale>1)&(summary.threads==2)].drop(columns=["threads"]),{"input_rows":"{:,}","median_seconds":"{:.4f}","iqr_seconds":"{:.4f}","median_peak_rss_mib":"{:.1f}","median_incremental_rss_mib":"{:.1f}"})

## 9. Cluster decision and handoff
The verified evidence supports a workload-specific decision about local processing. Compare measured 1× and synthetic 5×/10× performance against memory and service requirements. A multi-node deployment requires a separate measurement of communication, I/O, provisioning and cost. Part 4 reads the original verified Silver panel and separate dated event evidence; no synthetic rows, benchmark outputs or retrospective terminal status become model inputs.

Save the executed notebook with outputs to the public repository. Preserve evidence in the workspace; do not commit data files.